In [16]:
import csv
import re
from collections import Counter

import pandas as pd
from sklearn.metrics import classification_report
from text_filter import TextFilter
from text_normalizer import TextNormalizer

df = pd.read_csv("../../data/ruslan_dataset/metadata_RUSLAN_22200.csv", sep="|", header=None)
df.columns = ["audio_name", "raw text"]
df.head()

,audio_name,raw text
0,000000_RUSLAN,С тревожным чувством берусь я за перо.
1,000001_RUSLAN,Кого интересуют признания литературного неудач...
2,000002_RUSLAN,Что поучительного в его исповеди?
3,000003_RUSLAN,Да и жизнь моя лишена внешнего трагизма.
4,000004_RUSLAN,Я абсолютно здоров.


In [2]:
df["raw text length"] = df["raw text"].apply(len)
df["raw text words"] = df["raw text"].apply(lambda x: len(x.split()))
df.describe()

,raw text length,raw text words
count,22200.000000,22200.000000
mean,80.787297,12.027162
std,41.499187,6.229389
min,10.000000,1.000000
25%,55.000000,8.000000
50%,85.000000,12.000000
75%,102.000000,15.000000
max,723.000000,111.000000


# Ненормализованные примеры в корпусе

## Цифры

In [3]:
df[df["raw text"].str.contains(r"\d")]["raw text"].to_list()

['Брат разъезжал по отдаленным лагерным точкам. Ему предоставили казенную машину «ГАЗ‑61». ',
 'На Филиппинах кто‑то застрелил руководителя партийной оппозиции. Под Мелитополем разбился «ТУ‑129». ',
 'В сумочке ее лежало нечто, размером чуть поболее миниатюрного дамского браунинга «Элита‑16». ',
 '– Турбовинтовой МИ‑6, – заметил Пупс, вставая. – Е‑ё! – лениво крикнул он. Затем скрестил над головой руки. ']

## Английские слова

In [4]:
df[df["raw text"].str.contains(r"[A-Za-z]")]

,audio_name,raw text,raw text length,raw text words


## Пунктуация и смайлики

In [5]:
symbols = df[df["raw text"].str.contains(r"[°%₽$€]|&|@|\+|=")]
invalid_punctuation_combos = (
    r"(?<!\.)\.\.(?!\.)|"  # ровно две точки (не часть ...)
    r"\.{4,}|"  # 4 и более точек подряд
    r"!{2,}|"  # 2 и более восклицательных знака
    r"\?{2,}|"  # 2 и более вопросительных знака
    r"(?<!\.)[.,][.,?!…](?!\.)|"  # точка или запятая перед другими знаками
    r"\?[.,]|"  # вопросительный знак с точкой или запятой (?. или ?,)
    r"(?<!\?)[?!][.,]"  # восклицательный перед точкой/запятой, но разрешаем ?! перед точкой (?!.)
)
emoticons = r"[:;=]-?[pdзр3()\-\[\]]"

punc = df[df["raw text"].str.contains(invalid_punctuation_combos)]

print(f"Нестандартных символов: {len(symbols)}")
print(f"Примеров с эмотиконами: {int(df['raw text'].str.contains(emoticons).sum())}")
print(f"Примеров с неправильной пунктуацией: {len(punc)}")
punc.head()["raw text"].to_list()

Нестандартных символов: 0
Примеров с эмотиконами: 0
Примеров с неправильной пунктуацией: 628


['За что же моя рядовая, честная, единственная склонность подавляется бесчисленными органами, лицами, институтами великого государства??',
 'Вы страшное говно, мон колонель, не обессудьте!..',
 'Какать в одном поле не сяду!..',
 'Да это же писатель!..',
 'Расстреливать надо таких писателей!..']

## Междометия

In [6]:
stop_words = r"\b(?:ммм|гмм|хм|хе|хехе|хаха|мда|ыы|угу|ыых)\b"
stop = df[df["raw text"].str.contains(stop_words)]

print(f"Примеров с междометиями: {len(stop)}")

Примеров с междометиями: 0


## Абревиатуры, сокращения, и инициалы

По сути самая сложная в плане нормализации группа (поскольку английских слов нет, а для чисел мы возьмём num2words)

Возьмём за абревиатуру - все слова заглавные (как самый простой показатель)

In [7]:
initials = r"\b[А-ЯЁ]\.\s*(?:[А-ЯЁ]\.)?"
init = df[df["raw text"].str.contains(initials)]

print("Инициалы: ", len(init))
init.head()

Инициалы:  65


,audio_name,raw text,raw text length,raw text words
650,000650_RUSLAN,"А пиши ты, дурень, буквами А, Б, В.",35,8
748,000748_RUSLAN,О РАССКАЗАХ С. ДОВЛАТОВА,24,4
1041,001041_RUSLAN,при Обкоме вээлксэм В. Щербаков,31,5
1042,001042_RUSLAN,Члены литсекции В. Смирнов,26,4
1185,001185_RUSLAN,Звоню богачу Н. Предлагаю ему такой же вариант.,47,8


In [8]:
short_forms = (
    r"\b(?:г-жи|г-н)|"
    r"\b(?:ул|д|кв|г|прим)\."
)
short = df[df["raw text"].str.contains(short_forms)]
short.head()["raw text"].to_list()

['Уфлянда зовут Владимир прим. автора.',
 '…Контора размещалась тогда на улице Пикк. Строго напротив здания Госбезопасности (ул. Пагари, один). ',
 'Они – моя неврастения, злость, апломб, беспечность. И т. д. И самая кровавая война – бой призраков. ',
 'Я вынужден признать, что кто‑то из нас может стать первой жертвой этого критерия, но от этого он не становится менее надежным, а главное – менее единственным, если так можно говорить по‑русски… Мы уже говорили две литературы или одна – это, по сути дела, разговор о будущем литературы, эта литература едина, в ней есть элементы, которые присущи любой здоровой культуре, т. е. сосуществуют – реалистическая проза, авангард, наличествует сатира, нигилизм и т. д. В заключение я хочу сказать, что будущее литературы лежит, мне кажется, в сфере осознания литературой собственных прав и собственных возможностей.']

In [9]:
caps = r"\b[А-ЯЁ]{2,5}\b"
caps_mask = df["raw text"].str.contains(caps, regex=True, na=False)
all_caps_words = []
for text in df["raw text"]:
    all_caps_words.extend(re.findall(caps, text))

print(sorted(set(all_caps_words)))

['АН', 'АНУК', 'АРИКА', 'АТС', 'АХЧ', 'БАМ', 'БРЕМЯ', 'БРУНО', 'БТ', 'БУР', 'ВЕТРА', 'ВМК', 'ВОХРА', 'ВСЕ', 'ГАЗ', 'ГАИ', 'ГБ', 'ГЕЙНЕ', 'ГЛЯЖУ', 'ГОРОД', 'ГПУ', 'ГРОМ', 'ГУЛАГ', 'ДАР', 'ДЕЛОМ', 'ДЛЯ', 'ДОМА', 'ДПИ', 'ДЯР', 'ЕВРЕИ', 'ЕЛЕНА', 'ЕСТЬ', 'ЖИТЬ', 'ЗДЕСЬ', 'ЗМЕЙ', 'ИЗ', 'ИХ', 'КАК', 'КВВК', 'КВН', 'КП', 'КПП', 'КПСС', 'КПЭ', 'КТО', 'КЭМ', 'ЛГУ', 'ЛЕЩЬ', 'ЛИНДА', 'ЛИНДЕ', 'ЛИТМО', 'ЛИТО', 'ЛОМО', 'ЛОРЕН', 'ЛОСП', 'ЛУЧШЕ', 'МАРШ', 'МВД', 'МГУ', 'МИ', 'МОТИВ', 'МПВО', 'МТС', 'МЫ', 'НА', 'НАМ', 'НАРЯД', 'НЕ', 'НИИ', 'НИЦ', 'НКВД', 'НО', 'НОТ', 'ОВИР', 'ОДНИ', 'ОДНО', 'ОК', 'ОНИ', 'ООН', 'ПЕЙПС', 'ПЕРЕД', 'ПЛАНЫ', 'ПОПОВ', 'ПОРОК', 'ПОЧТИ', 'ПТУ', 'РАЙОН', 'РОМ', 'РЯДОМ', 'САМАЯ', 'СЕБЯ', 'СЕМЬ', 'СЛОВО', 'СНО', 'СОЛО', 'СС', 'ССР', 'СХШ', 'США', 'ТАМ', 'ТАСС', 'ТГУ', 'ТЕМ', 'ТПИ', 'ТУ', 'УВД', 'ФБР', 'ФД', 'ХУЖЕ', 'ЦДЛ', 'ЦО', 'ЧЕМ', 'ЧК', 'ЧУЖУЮ', 'ШВЕЙ', 'ШИЗО', 'ЭДАЗИ', 'ЭЛЛЕН', 'ЭМЕ', 'ЭТО', 'ЯЗЫКЕ']


капс не сработал. Выпишем оттуда аббревиатуры, которые читаются не как пишутся и возьмём 10 самых популярных для последующей замены

In [10]:
abbrev = "|".join(
    [
        "АТС",
        "АХЧ",
        "ВМК",
        "ГАЗ",
        "ГПУ",
        "ДОСААФ",
        "ДПИ",
        "КВН",
        "КПП",
        "КПСС",
        "КПЭ",
        "ЛГУ",
        "МВД",
        "МГУ",
        "НИЦ",
        "ПВО",
        "ПТУ",
        "СНО",
        "СССР",
        "ССР",
        "США",
        "ТГУ",
        "ТПИ",
        "УВД",
        "ФБР",
    ]
)
caps_mask = df["raw text"].str.contains(abbrev)
all_caps_words = []
for text in df["raw text"]:
    all_caps_words.extend(re.findall(abbrev, text))

Counter(all_caps_words).most_common(10)

[('КПСС', 6),
 ('СНО', 5),
 ('США', 4),
 ('УВД', 4),
 ('КПЭ', 3),
 ('ЛГУ', 3),
 ('МГУ', 3),
 ('ТГУ', 2),
 ('ВМК', 2),
 ('АХЧ', 2)]

# Обработка

In [18]:
normalyzer = TextNormalizer()
classifier = TextFilter()

dev_files = pd.read_csv(
    "data/dev_sentences.csv",
    sep="|",
    encoding="utf-8",
    quoting=csv.QUOTE_NONE,
    header=0,
)

dev_files["predicted"] = dev_files["text"].apply(classifier.filter)

print(classification_report(dev_files["is_normalized"], dev_files["predicted"], zero_division=0))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98       156
           1       0.97      0.99      0.98       144

    accuracy                           0.98       300
   macro avg       0.98      0.98      0.98       300
weighted avg       0.98      0.98      0.98       300



In [ ]:
total = df.shape[0]

start = sum(df["raw text"].apply(classifier.filter))

In [12]:
df["normalized text"] = df["raw text"].apply(normalyzer.normalize)
end = sum(df["normalized text"].apply(classifier.filter))

print(f"Всего предложений: {total}; предложений, проходящих фильтр: {start}; после нормализатора: {end}")
df.head()

Всего предложений: 22200; предложений, проходящих фильтр: 20415; после нормализатора: 21303


,audio_name,raw text,raw text length,raw text words,normalized text
0,000000_RUSLAN,С тревожным чувством берусь я за перо.,38,7,С тревожным чувством берусь я за перо.
1,000001_RUSLAN,Кого интересуют признания литературного неудач...,51,5,Кого интересуют признания литературного неудач...
2,000002_RUSLAN,Что поучительного в его исповеди?,33,5,Что поучительного в его исповеди?
3,000003_RUSLAN,Да и жизнь моя лишена внешнего трагизма.,40,7,Да и жизнь моя лишена внешнего трагизма.
4,000004_RUSLAN,Я абсолютно здоров.,19,3,Я абсолютно здоров.


In [ ]:
df[["audio_name", "raw text", "normalized text"]].to_csv(
    "../../data/ruslan_dataset/processed_ruslan.csv", sep="|", index=False
)

# Выводы

Мы взяли датасет Руслан, проанализировали его, отнормализовали согласно найденым правилам

## Анализ

1. Примеров с цифрами мало - спокойно удаляем через классификатор
2. Английских слов, эмотиконов, междометий и необычных символов - нет
3. Аббревиатуры вручную разметим 10 самых популярных, остальные будут удалены вместе с капс словами с помощью классификатора
4. Повторяющейся пунктуации много, но она спокойно приводится к нужному виду регексом, ровно как и стандартный набор сокращений

## Стратегия нормализатора

1. Приводим необычные юникод символы к стандартному набору
2. Разбираемся с аббревиатурами и сокращениями через словарь
3. Переводим числа в текст с помощью num2words
4. Избавляемся от повторяющейся пуктуации (в частности переводим многоточие в элипс символ для единства датасета)
5. Удаляем лишние пробелы

## Стратегия фильтра
1. Проверяем на наличие английских букв, цифр, сокращений, междометий и нестандартных символов
2. Проверяем менее стандартные сокращения
3. Zero-width spaces, которые нельзя найти на глаз
4. Нестандартные кавычки
5. Ещё сокращения
6. Не валидная пунктуация
7. Инициалы и эмотиконы
8. В конце проверяем на наличие отдельных слов полностью в верхнем регистре из 2 и более слов (чтобы не было FN на Я)